# Baghewala Field Digital Twin — Exploratory Data Analysis (EDA)
**Objective:** Analyze the 4 primary datasets across the production chain:
1. `data/raw/baghewala_css_dataset.csv` (10,000 CSS cycles, 20 wells)
2. `data/raw/train_rod.parquet` & `test_rod.parquet` (186,251 high-frequency SRP telemetry rows)
3. `data/raw/Volve production data.xlsx` (15,634 daily production records for analog decline benchmark)
4. Processed datasets in `data/processed/` and `data/external/`


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Current Directory:", os.getcwd())


### 1. Load and Inspect Baghewala CSS Dataset

In [ ]:
css_path = Path("data/raw/baghewala_css_dataset.csv")
df_css = pd.read_csv(css_path)
print(f"Baghewala CSS Dataset Shape: {df_css.shape}")
print(f"Unique Wells: {df_css['well_id'].nunique()}")
print(f"CSS Cycles Range: {df_css['css_cycle'].min()} to {df_css['css_cycle'].max()}")
df_css[['well_id', 'css_cycle', 'steam_injection_ton', 'oil_rate_bopd', 'steam_oil_ratio_ton_per_bbl', 'rod_floating_risk']].head()


### 2. Temperature Decline vs. Crude Viscosity Trajectory
Notice how as reservoir temperature cools from post-soak (~80°C) down to native reservoir baseline (46-48°C), heavy crude viscosity escalates from 3,000 cP up to >20,000 cP.

In [ ]:
temp_bins = pd.cut(df_css['end_production_temperature_c'], bins=8)
visc_trend = df_css.groupby(temp_bins)['estimated_viscosity_at_production_cp'].mean()
print("Temperature vs Average Viscosity (cP):")
print(visc_trend)


### 3. Sucker Rod Pump High-Frequency Telemetry
Inspect hourly dynacard indicators, load metrics, and pump fillage from `train_rod.parquet` & `test_rod.parquet`.

In [ ]:
pivoted_path = Path("data/processed/srp_telemetry_processed.parquet")
if pivoted_path.exists():
    df_telemetry = pd.read_parquet(pivoted_path)
    print(f"Pivoted Telemetry Shape: {df_telemetry.shape}")
    print(df_telemetry[['well_id', 'timestamp', 'spm', 'pprl_kn', 'mprl_kn', 'pump_fillage_pct', 'tubing_pressure_bar']].head())


### 4. Volve Analog Production Benchmark
Examine the North Sea conventional sandstone baseline for decline curve comparison.

In [ ]:
volve_proc = Path("data/processed/volve_daily_production_processed.parquet")
if volve_proc.exists():
    df_volve = pd.read_parquet(volve_proc)
    print(f"Volve Cleaned Shape: {df_volve.shape}")
    print(f"Producing Wells: {df_volve['well_bore_code'].nunique()}")
    print(f"Mean Oil Rate: {df_volve[df_volve['bore_oil_vol'] > 0]['bore_oil_vol'].mean():.2f} m3/day")
